In [3]:
import sys
!{sys.executable} -m pip install librosa --break-system-packages

  Using cached librosa-0.11.0-py3-none-any.whl.metadata (8.7 kB)
  Using cached audioread-3.1.0-py3-none-any.whl.metadata (9.0 kB)
  Using cached soundfile-0.13.1-py2.py3-none-macosx_11_0_arm64.whl.metadata (16 kB)
  Using cached pooch-1.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached soxr-1.0.0-cp312-abi3-macosx_11_0_arm64.whl.metadata (5.6 kB)
Using cached librosa-0.11.0-py3-none-any.whl (260 kB)
Using cached audioread-3.1.0-py3-none-any.whl (23 kB)
Using cached pooch-1.9.0-py3-none-any.whl (67 kB)
Using cached soundfile-0.13.1-py2.py3-none-macosx_11_0_arm64.whl (1.1 MB)
Using cached soxr-1.0.0-cp312-abi3-macosx_11_0_arm64.whl (163 kB)


In [4]:
import pandas as pd
import numpy as np
import librosa
import librosa.display
import os

In [5]:
df = pd.read_csv('../data/cleaned/fma_cleaned_dataset.csv')


In [6]:
 ## Parameters 
SAMPLE_RATE = 22050
DURATION = 30       
N_MELS = 128         
HOP_LENGTH = 512
N_FFT = 2048

In [7]:
def mp3_to_spectrogram(mp3_path, sr=SAMPLE_RATE, duration=DURATION):
    """
    Loads an MP3 file and converts it to a mel spectrogram.
    Returns a 2D numpy array (n_mels x time_frames), or None if the file fails.
    """
    try:
        # Loading the audio
        y, sr = librosa.load(mp3_path, sr=sr, duration=duration, mono=True)

        # Computing mel spectrogram
        mel_spec = librosa.feature.melspectrogram(
            y=y,
            sr=sr,
            n_mels=N_MELS,
            n_fft=N_FFT,
            hop_length=HOP_LENGTH
        )

        # Converting to log scale 
        mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)

        # Normalizing to [0, 1]
        mel_spec_norm = (mel_spec_db - mel_spec_db.min()) / (mel_spec_db.max() - mel_spec_db.min())

        return mel_spec_norm

    except Exception as e:
        print(f"Failed to process {mp3_path}: {e}")
        return None

In [8]:
## Generating and saving spectrograms
spectrograms = []
labels = []
track_ids = []

for _, row in df.iterrows():
    spec = mp3_to_spectrogram(row['mp3_path'])
    if spec is not None:
        spectrograms.append(spec)
        labels.append(row['genre_top'])
        track_ids.append(row['track_id'])

In [9]:
## Stacking into arrays
spectrograms = np.array(spectrograms)   
labels = np.array(labels)
track_ids = np.array(track_ids)

print(f"Spectrogram array shape: {spectrograms.shape}")
print(f"Labels: {np.unique(labels)}")

np.save('../data/cleaned/fma_spectrograms.npy', spectrograms)
np.save('../data/cleaned/fma_labels.npy', labels)
np.save('../data/cleaned/fma_track_ids.npy', track_ids)

print("Saved spectrograms, labels, and track IDs.")

Spectrogram array shape: (0,)
Labels: []
Saved spectrograms, labels, and track IDs.
